![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

# Using BUFR

In this notebook you will see how to:

- inspect BUFR data
- extract BUFR data into a Pandas dataframe

## Getting the data

First we read some [BUFR](https://pdbufr.readthedocs.io/en/latest/guide/bufr_primer.html) data from disk with [from_source](https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html). 

In [1]:
import earthkit.data as ekd

d = ekd.from_source("sample", "synop_10.bufr")
d

synop_10.bufr:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

path,/var/folders/93/w0p869rx17q98wxk83gn9ys40000gn/T/tmptsch0oj5/url-0fad6f72ff51300f25405332acfb47cac22e36112667f7067cc7a7bebde08477.bufr
size,2.1 KiB
types,"pandas, featurelist"


### Featurelists and BUFR messages

To inspect BUFR data we need to convert it into a featureslist. It is a similar object to a fieldList, but it is an iterable of "features", where a "feature" can be anything. In a BUFR featurelist each feature is a [BUFR message](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/message/index.html).

In [2]:
fl = d.to_featurelist()

The file contains 10 messages.

In [3]:
len(fl)

10

With the [ls()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/file/index.html#earthkit.data.readers.bufr.file.BUFRList.ls) method we can get a summary of the messages (using header keys).

In [4]:
fl.ls()

,edition,dataCategory,dataSubCategory,bufrHeaderCentre,masterTablesVersionNumber,localTablesVersionNumber,numberOfSubsets,compressedData,typicalDate,typicalTime,ident,localLatitude,localLongitude
0,3,0,1,98,13,1,1,0,20230602,120000,91648,-10.75,179.50
1,3,0,1,98,13,1,1,0,20230602,120000,89514,-70.77,11.75
2,3,0,1,98,13,1,1,0,20230602,120000,60545,33.77,2.93
3,3,0,1,98,13,1,1,0,20230602,120000,30823,51.83,107.60
4,3,0,1,98,13,1,1,0,20230602,120000,30846,51.35,112.47
5,3,0,1,98,13,1,1,0,20230602,120000,48352,17.86,102.75
6,3,0,1,98,13,1,1,0,20230602,120000,98747,8.41,124.61
7,3,0,1,98,13,1,1,0,20230602,120000,68267,-26.50,29.98
8,3,0,1,98,13,1,1,0,20230602,120000,68592,-29.60,31.12
9,3,0,1,98,13,1,1,0,20230602,120000,91701,-2.77,-171.72


#### BUFR messages

In [5]:
# f is the first message in the featurelist
f = fl[0]
f

BUFRMessage(type=0,subType=1,subsets=1,20230602,120000)

To dump the contents of a message (as a tree view) use [describe()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/message/index.html#earthkit.data.readers.bufr.message.BUFRMessage.describe).

In [6]:
f.describe()

### Subsetting

With [sel()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/file/index.html#earthkit.data.readers.bufr.file.BUFRList.sel)
we can select the messages matching the given metadata conditions.

In [7]:
fl1 = fl.sel(dataSubCategory=1, ident=[60545, 48352])
fl1.ls()

,edition,dataCategory,dataSubCategory,bufrHeaderCentre,masterTablesVersionNumber,localTablesVersionNumber,numberOfSubsets,compressedData,typicalDate,typicalTime,ident,localLatitude,localLongitude
0,3,0,1,98,13,1,1,0,20230602,120000,60545,33.77,2.93
1,3,0,1,98,13,1,1,0,20230602,120000,48352,17.86,102.75


### Converting to pandas

BUFR data can be extracted into a Pandas dataframe using [to_pandas()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/file/index.html#earthkit.data.readers.bufr.file.BUFRList.to_pandas), which passes all the arguments to the [read_bufr()](https://pdbufr.readthedocs.io/en/latest/guide/read_bufr.html) method from ``pdbufr``.

In [8]:
df = fl.to_pandas(columns=["latitude", "longitude",
                           "heightOfStation","airTemperatureAt2M"])
df

,latitude,longitude,heightOfStation,airTemperatureAt2M
0,-10.75,179.50,3.0,300.4
1,-70.77,11.75,NaN,255.2
2,33.77,2.93,763.0,296.3
3,51.83,107.60,515.0,291.6
4,51.35,112.47,743.0,287.4
5,17.86,102.75,176.0,307.9
6,8.41,124.61,188.0,299.4
7,-26.50,29.98,1774.0,281.9
8,-29.60,31.12,105.0,299.8
9,-2.77,-171.72,2.0,302.1


In [9]:
df = fl.to_pandas(reader="synop")
df

,stnid,lat,lon,elevation,time,t2m,rh2m,td2m,wind10m_speed,wind10m_dir,...,present_weather,precipitation_6h,precipitation_12h,min_t2m,max_t2m,mslp,pressure,cloud_cover,snow_depth,visibility
0,91648,-10.75,179.50,3.0,2023-06-02 12:00:00,300.4,None,298.1,4.0,80,...,2,0.0,NaN,None,None,101220.0,NaN,60,None,55000.0
1,89514,-70.77,11.75,NaN,2023-06-02 12:00:00,255.2,None,NaN,12.0,180,...,2,NaN,0.0,None,None,98780.0,NaN,40,None,10000.0
2,60545,33.77,2.93,763.0,2023-06-02 12:00:00,296.3,None,285.0,5.0,60,...,2,NaN,0.0,None,None,NaN,92720.0,90,None,8000.0
3,30823,51.83,107.60,515.0,2023-06-02 12:00:00,291.6,None,277.2,4.0,310,...,1,NaN,0.0,None,None,101640.0,95740.0,40,None,55000.0
4,30846,51.35,112.47,743.0,2023-06-02 12:00:00,287.4,None,277.7,4.0,230,...,1,NaN,0.6,None,None,101460.0,92970.0,75,None,55000.0
5,48352,17.86,102.75,176.0,2023-06-02 12:00:00,307.9,None,298.5,2.0,140,...,2,NaN,0.0,None,None,100190.0,98260.0,25,None,12000.0
6,98747,8.41,124.61,188.0,2023-06-02 12:00:00,299.4,None,296.7,1.0,150,...,5,0.0,NaN,None,None,100980.0,100120.0,10,None,15000.0
7,68267,-26.50,29.98,1774.0,2023-06-02 12:00:00,281.9,None,274.3,5.0,310,...,2,0.0,NaN,None,None,NaN,82650.0,0,None,20000.0
8,68592,-29.60,31.12,105.0,2023-06-02 12:00:00,299.8,None,293.3,6.0,40,...,2,0.0,NaN,None,None,101260.0,100220.0,0,None,20000.0
9,91701,-2.77,-171.72,2.0,2023-06-02 12:00:00,302.1,None,NaN,3.0,70,...,2,0.0,NaN,None,None,101190.0,NaN,40,None,55000.0


Please note it is also possible to call [to_pandas()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/readers/bufr/file/index.html#earthkit.data.readers.bufr.file.BUFRList.to_pandas) on the input data object "d". It this case, first the data is converted to a featurelist under the hood, then ``to_pandas()`` is called on the featurelist.

In [10]:
df = d.to_pandas(columns=["latitude", "longitude",
                           "heightOfStation","airTemperatureAt2M"])
df

,latitude,longitude,heightOfStation,airTemperatureAt2M
0,-10.75,179.50,3.0,300.4
1,-70.77,11.75,NaN,255.2
2,33.77,2.93,763.0,296.3
3,51.83,107.60,515.0,291.6
4,51.35,112.47,743.0,287.4
5,17.86,102.75,176.0,307.9
6,8.41,124.61,188.0,299.4
7,-26.50,29.98,1774.0,281.9
8,-29.60,31.12,105.0,299.8
9,-2.77,-171.72,2.0,302.1


### Extra work

Get the "temp_10.bufr" radiosonde BUFR file as a sample and extract the latitude-longitude and the pressure-temperature profile from it for station "01415". 

Hints:
- use the "indent" key for the station ID
- use the "filters" kwarg in ``to_pandas()`` (for details about the format see [here](https://pdbufr.readthedocs.io/en/latest/guide/misc/filters.html#filters))
- use the "pressure" and "airTemperature" keys (use ``describe()`` to inspect the radiosonde message structure)

In [11]:
fl = ekd.from_source("sample", "temp_10.bufr").to_featurelist()
fl.ls()

temp_10.bufr:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

,edition,dataCategory,dataSubCategory,bufrHeaderCentre,masterTablesVersionNumber,localTablesVersionNumber,numberOfSubsets,compressedData,typicalDate,typicalTime,ident,localLatitude,localLongitude
0,3,2,101,98,13,1,1,0,20081208,120000,02836,67.37,26.63
1,3,2,101,98,13,1,1,0,20081208,120000,01400,56.90,3.35
2,3,2,101,98,13,1,1,0,20081208,120000,01415,58.87,5.67
3,3,2,101,98,13,1,1,0,20081208,120000,01001,70.93,-8.67
4,3,2,101,98,13,1,1,0,20081208,120000,01152,67.28,14.45
5,3,2,101,98,13,1,1,0,20081208,120000,01241,63.70,9.60
6,3,2,101,98,13,1,1,0,20081208,120000,03953,51.93,-10.25
7,3,2,101,98,13,1,1,0,20081208,120000,11747,49.45,17.13
8,3,2,101,98,13,1,1,0,20081208,120000,11035,48.25,16.37
9,3,2,101,98,13,1,1,0,20081208,120000,02963,60.82,23.50


In [12]:
fl[0].describe()

In [13]:
df = fl.to_pandas(columns=["latitude", "longitude",
                           "pressure","airTemperature"],
                  filters={"ident": "01415"})
df

,latitude,longitude,pressure,airTemperature
0,58.87,5.67,100300.0,279.8
1,58.87,5.67,100000.0,280.0
2,58.87,5.67,98900.0,NaN
3,58.87,5.67,97800.0,NaN
4,58.87,5.67,92500.0,275.2
...,...,...,...,...
58,58.87,5.67,1840.0,NaN
59,58.87,5.67,1790.0,197.5
60,58.87,5.67,1500.0,193.1
61,58.87,5.67,1380.0,NaN
